# 22 — What gets challenged (2026)

Challenged pitches by role, count, inning, inventory, pitch type and
location. Tests two hypotheses from notebooks 20 and 21: that batters
drive the full-count collapse, and that batters spend the expiring
challenge late. Sources: `challenges_enriched`, `challenges_remaining`,
`pitches` (2026). Every rate carries a 95% Wilson interval.

In [ ]:
import os
import sys

import pandas as pd

sys.path.insert(0, "../src")
from stats_helpers import rate_table

pd.set_option("display.width", 130)
pd.set_option("display.max_rows", 60)
TABLES = "../data/tables"
REPORTS = "../reports/tables"
KEY = ["game_pk", "at_bat_index", "pitch_number"]

chal = pd.read_parquet(f"{TABLES}/challenges_enriched/all_seasons.parquet")
rem = pd.read_parquet(f"{TABLES}/challenges_remaining/season=2026/part.parquet")
pit = pd.read_parquet(
    f"{TABLES}/pitches/season=2026/part.parquet",
    columns=KEY + ["balls_after", "strikes_after", "pitch_type_raw",
                   "start_speed_mph", "plate_x_ft", "plate_z_ft",
                   "sz_top_ft", "sz_bot_ft", "bat_side"],
)
c = chal.merge(rem, on=KEY, how="left").merge(pit, on=KEY, how="left")

c["balls_pre"] = c["balls_after"] - (c["final_call_code"] == "B").astype(int)
c["strikes_pre"] = c["strikes_after"] - (c["final_call_code"] == "C").astype(int)
c["count_pre"] = c["balls_pre"].astype(str) + "-" + c["strikes_pre"].astype(str)
c["left_before"] = c["challenges_left_home"].where(
    c["challenger_side"] == "home", c["challenges_left_away"])
c["late"] = c["inning"] >= 9
c["vertical"] = pd.cut(
    (c["plate_z_ft"] - c["sz_bot_ft"]) / (c["sz_top_ft"] - c["sz_bot_ft"]),
    bins=[-10, 0, 0.33, 0.67, 1, 10],
    labels=["below", "low", "middle", "high", "above"],
)
print(len(c), "challenges;", c["plate_x_ft"].isna().sum(), "without location")

In [ ]:
t = rate_table(c, ["challenger_role", "count_pre"], "is_overturned", min_n=50)
t = t[t["challenger_role"].isin(["batter", "catcher"])]
display(t.pivot(index="count_pre", columns="challenger_role", values="rate").round(3))
display(t.pivot(index="count_pre", columns="challenger_role", values="trials"))

In [ ]:
t2 = rate_table(c[c["challenger_role"].isin(["batter", "catcher"])],
                ["challenger_role", "late", "left_before"], "is_overturned")
display(t2.sort_values(["challenger_role", "late", "left_before"]))

In [ ]:
os.makedirs(REPORTS, exist_ok=True)
t.to_csv(f"{REPORTS}/challenges_role_by_count_2026.csv", index=False)
t2.to_csv(f"{REPORTS}/challenges_role_by_late_inventory_2026.csv", index=False)
print("saved")

In [ ]:
print("--- by pitch type (min 100) ---")
display(rate_table(c, ["pitch_type_raw"], "is_overturned", min_n=100).head(12))
print("--- by vertical location relative to the batter's zone ---")
display(rate_table(c, ["vertical"], "is_overturned").sort_values("vertical"))
print("--- by original call ---")
display(rate_table(c, ["original_call_code", "vertical"], "is_overturned")
        .sort_values(["original_call_code", "vertical"]))

In [ ]:
rate_table(c, ["pitch_type_raw"], "is_overturned", min_n=100).to_csv(
    f"{REPORTS}/challenges_by_pitch_type_2026.csv", index=False)
rate_table(c, ["original_call_code", "vertical"], "is_overturned").to_csv(
    f"{REPORTS}/challenges_by_call_and_location_2026.csv", index=False)
print("saved")